# MMELON Single-View Fine-Tuning with K-Fold Cross-Validation

Fine-tune one encoder of the pretrained MMELON model (text, image or graph) on its own for **binary classification**
on every fold of a k-fold split, and select the final model by the **average metric across folds**.

**Environment:** run with the `biomed-multiview` kernel.

**Single view:** trains one MMELON encoder on its own, with no fusion: `VIEW = "text"` (SMILES transformer,
MoLFormer architecture), `"image"` (ResNet18, ImageMol) or `"graph"` (graph transformer, TokenGT). The encoder
starts from its weights in the pretrained MMELON checkpoint (`ibm-research/biomed.sm.mv-te-84m`) and is
followed by the same MLP prediction head as MMELON. Everything else (inputs, metrics, model selection, outputs)
is the same as the multi-view notebooks, so results are directly comparable.
For the multi-view model (MMELON), use the matching `MMELON_*` notebook without `singleview` in its name.

**Inputs**
- A CSV with a SMILES column and a 0/1 label column.
- An `.npz` file of fold indices, as made by [stratified_cv.py](stratified_cv.py)
  (arrays `fold0_train`, `fold0_test`, `fold1_train`, ...). The number of folds is read from this file.

**Metrics**
- **hit@K** (hit@200, hit@100, hit@50): rank the test molecules by predicted score; hit@K is how many
  of the top K have label 1. If a test set has fewer than K molecules, all of them are counted.
- Standard classification metrics: AUROC, precision, recall, specificity and loss.

**How the model is selected**
1. For each fold, train on the fold's train rows for `EPOCHS` epochs, scoring the fold's test rows after every epoch.
2. Average each epoch's metrics over all folds.
3. Rank epochs by `SELECT_METRICS` (default: hit@200, then hit@100, then hit@50, then AUROC to break ties).
   Each fold's checkpoint from the best epoch is kept (K models, usable as an ensemble);
   checkpoints from other epochs are deleted.

Because the test folds are used to pick the epoch, the selected average is slightly optimistic.
For an unbiased estimate, evaluate the final models on data not in the CSV.

**Output** (in the gitignored `aircheck/output/`):
```
output/
├── base_models/biomed.sm.mv-te-84m/      # pretrained model (downloaded once)
└── <RUN_NAME>_<VIEW>/
    ├── config.json                       # settings used
    ├── cv_metrics_per_epoch.csv          # every fold × epoch
    ├── cv_summary_per_epoch.csv          # mean and std over folds per epoch
    ├── selection.json                    # selected epoch, its scores, model paths
    └── fold_0/, fold_1/, ...
        ├── data/                         # data_train.csv, data_val.csv (= this fold's test rows)
        ├── logs/metrics.csv
        └── best_model.ckpt               # this fold's model at the selected epoch
```

## 1. Inputs and settings

**Edit these before running.** Data files can be `.csv` or `.parquet`; only the columns the notebook needs are read.

In [ ]:
from pathlib import Path

# Name of this run; outputs go to output/<RUN_NAME>_<VIEW>/ (must not already exist)
RUN_NAME = "sample-cv"

# Encoder to train on its own: "text" (SMILES transformer, MoLFormer), "image" (ResNet18, ImageMol)
# or "graph" (graph transformer, TokenGT)
VIEW = "text"

# ── Data ──────────────────────────────────────────────────────────────────────
DATA_DIR = Path("../output/cv_folds")   # made by cluster_cv_folds.ipynb

DATA_PATH = DATA_DIR / "train_cluster_cv_data.csv"
CV_INDICES_PATH = DATA_DIR / "train_cluster_cv_2fold_indices.npz"

SMILES_COL = "SMILES"
LABEL_COL = "LABEL"

In [ ]:
import os

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model ID

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS = 10               # epochs per fold; the best epoch is chosen across folds
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42

# hit@K = number of label-1 molecules among the K highest-scored test molecules
HIT_KS = [200, 100, 50]

# Model selection, in priority order: the first metric decides, later ones break ties.
# Choose from hit@K (for K in HIT_KS), "rocauc", "precision", "recall", "specificity", "loss".
SELECT_METRICS = ["hit@200", "hit@100", "hit@50", "rocauc"]

NUM_WORKERS = min(4, os.cpu_count())

In [ ]:
import json
import shutil

import numpy as np
import pandas as pd
import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bmfm_sm").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-view repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "aircheck" / "output"
BASE_MODEL_DIR = OUTPUT_DIR / "base_models" / BASE_MODEL.split("/")[-1]
RUN_DIR = OUTPUT_DIR / f"{RUN_NAME}_{VIEW}"

assert VIEW in ("text", "image", "graph"), f"VIEW must be text, image or graph (for multiview use MMELON_finetuning_cv.ipynb), got {VIEW!r}"
for path in (DATA_PATH, CV_INDICES_PATH):
    assert Path(path).exists(), f"File not found: {path}"
assert not (RUN_DIR.exists() and any(RUN_DIR.iterdir())), (
    f"{RUN_DIR} already exists. Choose a new RUN_NAME or delete that folder."
)

# ── Fold indices ──────────────────────────────────────────────────────────────
cv_indices = np.load(CV_INDICES_PATH)
N_FOLDS = len([k for k in cv_indices.files if k.endswith("_train")])
folds = [(cv_indices[f"fold{i}_train"], cv_indices[f"fold{i}_test"]) for i in range(N_FOLDS)]
for i, (train_idx, test_idx) in enumerate(folds):
    assert not np.intersect1d(train_idx, test_idx).size, f"fold {i}: train and test rows overlap"

# Each fold keeps one checkpoint per epoch (~0.35 GB) until the best epoch is chosen
needed_gb = 0.35 * N_FOLDS * EPOCHS
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > needed_gb + 2, f"Need ~{needed_gb:.0f} GB free for checkpoints, only {free_gb:.0f} GB available"

RUN_DIR.mkdir(parents=True, exist_ok=True)
config = dict(
    data_path=str(DATA_PATH), cv_indices_path=str(CV_INDICES_PATH), n_folds=N_FOLDS,
    smiles_col=SMILES_COL, label_col=LABEL_COL, base_model=BASE_MODEL, epochs=EPOCHS,
    batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    seed=SEED, hit_ks=HIT_KS, select_metrics=SELECT_METRICS, view=VIEW,
)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Folds      : {N_FOLDS} (from {CV_INDICES_PATH.name})")
print(f"Run folder : {RUN_DIR}")
print(f"View       : {VIEW}")
print(f"Disk       : ~{needed_gb:.1f} GB needed during training, {free_gb:.0f} GB free")
print(f"Device     : {'GPU' if torch.cuda.is_available() else 'CPU (training will be slow)'}")

## 2. Prepare data

Loads the CSV and writes each fold's rows in the format the MMELON data pipeline expects
(`data_train.csv`, and `data_val.csv` for the rows scored after each epoch).

In [ ]:
import pyarrow.parquet as pq


def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)


df = read_table(DATA_PATH, [SMILES_COL, LABEL_COL]).rename(columns={SMILES_COL: "smiles", LABEL_COL: "label"})

# Rows are referenced by position, so bad rows are reported rather than dropped
bad = df["smiles"].isna() | (df["smiles"].astype(str).str.strip() == "") | df["label"].isna()
assert not bad.any(), f"{bad.sum()} rows have empty SMILES or label, e.g. rows {df.index[bad][:5].tolist()}"
df["label"] = df["label"].astype(int)
assert set(df["label"].unique()) <= {0, 1}, "labels must be 0/1"

max_index = max(int(idx.max()) for pair in folds for idx in pair)
assert max_index < len(df), f"Fold indices go up to {max_index} but {DATA_PATH.name} has {len(df)} rows"

summary = []
for i, (train_idx, test_idx) in enumerate(folds):
    data_dir = RUN_DIR / f"fold_{i}" / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    df.iloc[train_idx].to_csv(data_dir / "data_train.csv", index=False)
    df.iloc[test_idx].to_csv(data_dir / "data_val.csv", index=False)
    summary.append(dict(
        fold=i, train=len(train_idx), test=len(test_idx),
        train_pos_pct=round(100 * df["label"].iloc[train_idx].mean(), 2),
        test_pos_pct=round(100 * df["label"].iloc[test_idx].mean(), 2),
    ))

print(f"{len(df):,} rows, {100 * df['label'].mean():.2f}% positive")
pd.DataFrame(summary)

## 3. Download the base model

Downloaded once into `output/base_models/`; later runs reuse it.

In [ ]:
from huggingface_hub import snapshot_download

if (BASE_MODEL_DIR / "model.safetensors").exists():
    print(f"Using existing base model: {BASE_MODEL_DIR}")
else:
    print(f"Downloading {BASE_MODEL} ...")
    snapshot_download(repo_id=BASE_MODEL, local_dir=BASE_MODEL_DIR)
    print(f"Saved to: {BASE_MODEL_DIR}")

## 4. Model and data helpers

In [ ]:
import gc
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.WARNING)

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import DataLoader

from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel
from bmfm_sm.core.data_modules.namespace import TaskType
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule

from bmfm_sm.core.data_modules.namespace import LateFusionStrategy
from bmfm_sm.predictive.data_modules.graph_finetune_dataset import Graph2dFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.image_finetune_dataset import ImageFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.text_finetune_dataset import TextFinetuneDataPipeline

# Per view: model class and settings, data pipeline, input modality, and the prefix of that
# view's encoder weights inside the MMELON checkpoint
VIEW_SPECS = {
    "text": dict(
        model_class="bmfm_sm.predictive.modules.text_models.TextModel",
        model_params={},
        pipeline=TextFinetuneDataPipeline,
        modalities=["TEXT_MODEL"],
        weight_prefix="model_text.",
    ),
    "image": dict(
        model_class="bmfm_sm.predictive.modules.image_models.ImageModel",
        model_params={"baseModel": "ResNet18"},
        pipeline=ImageFinetuneDataPipeline,
        modalities=["IMAGE_MODEL"],
        weight_prefix="model_image.",
    ),
    "graph": dict(
        model_class="bmfm_sm.predictive.modules.graph_2d_models.Graph2dModel",
        model_params={"encoder_embed_dim": 512},
        pipeline=Graph2dFinetuneDataPipeline,
        modalities=["GRAPH_2D_MODEL"],
        weight_prefix="model_graph.",
    ),
}


def dataset_args_for(view: str) -> dict:
    return {
        "task_type": TaskType.CLASSIFICATION,
        "num_tasks": 1,
        "modalities": VIEW_SPECS[view]["modalities"],
        "smiles_col": "smiles",
        "label_cols": ["label"],
        "split_col": None,
    }


def view_of_checkpoint(path) -> str:
    # Which view a fine-tuned checkpoint was trained on, from the model class saved in it
    model_class = torch.load(path, map_location="cpu")["hyper_parameters"]["base_model_class"]
    matches = [v for v, s in VIEW_SPECS.items() if s["model_class"] == model_class]
    assert matches, f"{path} is not a single-view model ({model_class}); use the MMELON_* notebooks for multiview"
    return matches[0]


def to_device(batch):
    # Batches are dicts (multiview, graph) or tuples (text, image)
    if isinstance(batch, dict):
        return {k: to_device(v) for k, v in batch.items()}
    if isinstance(batch, (list, tuple)):
        return type(batch)(to_device(v) for v in batch)
    return batch.to(DEVICE) if torch.is_tensor(batch) else batch

SPEC = VIEW_SPECS[VIEW]

# Load the pretrained weights once; every fold starts from a copy of them.
# Only this view's encoder is used (weight names without the view prefix).
prefix = SPEC["weight_prefix"]
pretrained_state = {
    k[len(prefix):]: v
    for k, v in SmallMoleculeMultiViewModel.from_pretrained(
        LateFusionStrategy.ATTENTIONAL, model_path=str(BASE_MODEL_DIR), huggingface=True
    ).state_dict().items()
    if k.startswith(prefix)
}


def make_loader(data_dir: Path, stage: str, shuffle: bool) -> DataLoader:
    dataset = SPEC["pipeline"](data_dir=str(data_dir), dataset_args=dataset_args_for(VIEW), stage=stage)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=dataset.collate_fn, num_workers=NUM_WORKERS)


class HitAtKFineTuneModule(FineTuneLightningModule):
    # Adds hit@K to the scores logged after every epoch (as val_hit@K)

    def on_validation_epoch_start(self):
        super().on_validation_epoch_start()
        self._val_scores, self._val_labels = [], []

    def validation_step(self, batch, batch_idx):
        loss, y, y_hat = self._shared_step(batch, "val")
        self._val_scores.append(y_hat.detach().float().reshape(-1).cpu())
        self._val_labels.append(y.detach().reshape(-1).cpu())
        return loss

    def on_validation_epoch_end(self):
        super().on_validation_epoch_end()
        scores, labels = torch.cat(self._val_scores), torch.cat(self._val_labels)
        top = torch.argsort(scores, descending=True)
        for k in HIT_KS:
            self.log(f"val_hit@{k}", float((labels[top[:k]] == 1).sum()))


def make_model() -> HitAtKFineTuneModule:
    module = HitAtKFineTuneModule(
        base_model_class=SPEC["model_class"],
        model_params=SPEC["model_params"],
        task_type="classification",
        num_tasks=1,
        checkpoint_path=None,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        finetuning_args={
            "weight_freeze": "unfrozen",
            "initialization": "default",
            "head_arch": "mlp",
            "use_norm": True,
            "head_dropout": 0.2,
        },
    )
    result = module.model.load_state_dict(pretrained_state, strict=False)
    # strict=False would silently skip mismatched weights, so check explicitly
    assert not result.missing_keys and not result.unexpected_keys, (
        f"Pretrained weights did not match: missing={result.missing_keys[:5]}, "
        f"unexpected={result.unexpected_keys[:5]}"
    )
    return module


def epoch_checkpoint(fold: int, epoch: int) -> Path:
    return RUN_DIR / f"fold_{fold}" / "checkpoints" / f"epoch{epoch:02d}.ckpt"


def order_metrics(columns) -> list:
    # hit@K first (in HIT_KS order), then the standard metrics
    hits = [f"hit@{k}" for k in HIT_KS]
    return [c for c in hits if c in columns] + [c for c in columns if c not in hits]


print(f"Helpers ready; pretrained {VIEW} weights loaded.")

## 5. Train every fold

Each fold saves a weights-only checkpoint after every epoch, so any epoch can be selected afterwards.
Metrics prefixed `val_` in the logs are scores on that fold's **test** rows.

In [ ]:
%%time
fold_metrics = []

for fold in range(N_FOLDS):
    print("=" * 80)
    print(f"FOLD {fold + 1}/{N_FOLDS}")
    print("=" * 80)
    fold_dir = RUN_DIR / f"fold_{fold}"
    pl.seed_everything(SEED + fold)

    train_loader = make_loader(fold_dir / "data", "train", shuffle=True)
    test_loader = make_loader(fold_dir / "data", "val", shuffle=False)
    module = make_model()

    trainer = pl.Trainer(
        max_epochs=EPOCHS,
        callbacks=[ModelCheckpoint(
            dirpath=fold_dir / "checkpoints",
            filename="epoch{epoch:02d}",
            auto_insert_metric_name=False,
            save_top_k=-1,              # keep every epoch until selection
            every_n_epochs=1,
            save_weights_only=True,
        )],
        logger=CSVLogger(save_dir=fold_dir, name="logs", version=""),
        default_root_dir=fold_dir,
        accelerator="auto",
        devices=1,
        log_every_n_steps=10,
        num_sanity_val_steps=0,
        enable_model_summary=False,
    )
    trainer.fit(module, train_loader, test_loader)

    logged = pd.read_csv(fold_dir / "logs" / "metrics.csv")
    epochs_df = logged.dropna(subset=["val_loss"])
    metric_cols = [c for c in epochs_df.columns if c.startswith("val_")]
    epochs_df = epochs_df[["epoch"] + metric_cols].rename(columns=lambda c: c.removeprefix("val_"))
    epochs_df = epochs_df.groupby("epoch", as_index=False).first()  # one row per epoch
    fold_metrics.append(epochs_df.assign(fold=fold))

    del trainer, module, train_loader, test_loader
    gc.collect()

cv_metrics = pd.concat(fold_metrics, ignore_index=True)
cv_metrics["epoch"] = cv_metrics["epoch"].astype(int)
metric_names = order_metrics([c for c in cv_metrics.columns if c not in ("fold", "epoch")])
cv_metrics = cv_metrics[["fold", "epoch"] + metric_names]
cv_metrics.to_csv(RUN_DIR / "cv_metrics_per_epoch.csv", index=False)

## 6. Select the best epoch across folds

Epochs are ranked by the fold-averaged `SELECT_METRICS` in order: the first metric decides,
and each later one breaks ties (e.g. two epochs with the same mean hit@200 are compared on hit@100).

In [ ]:
missing = [m for m in SELECT_METRICS if m not in metric_names]
assert not missing, f"SELECT_METRICS {missing} not logged; available: {metric_names}"

cv_summary = cv_metrics.groupby("epoch")[metric_names].agg(["mean", "std"])
cv_summary.columns = [f"{m}_{stat}" for m, stat in cv_summary.columns]
cv_summary = cv_summary.reset_index()

# Only epochs that every fold completed are eligible
complete = cv_metrics.groupby("epoch")["fold"].nunique() == N_FOLDS
cv_summary = cv_summary[cv_summary["epoch"].isin(complete[complete].index)]
cv_summary.to_csv(RUN_DIR / "cv_summary_per_epoch.csv", index=False)

# Sort by each selection metric in priority order (loss: lower is better); ties keep the earlier epoch
ranked = cv_summary.sort_values(
    [f"{m}_mean" for m in SELECT_METRICS],
    ascending=[m == "loss" for m in SELECT_METRICS],
    kind="mergesort",
)
best_row = ranked.iloc[0]
BEST_EPOCH = int(best_row["epoch"])

print(f"Selected epoch {BEST_EPOCH} (mean over {N_FOLDS} folds):")
for m in SELECT_METRICS:
    print(f"  {m:12s} {best_row[f'{m}_mean']:.4f}  (std {best_row[f'{m}_std']:.4f})")
display(cv_summary.round(4))
display(cv_metrics[cv_metrics["epoch"] == BEST_EPOCH].round(4).reset_index(drop=True))

## 7. Keep the selected models

Each fold's checkpoint from the selected epoch becomes `fold_<i>/best_model.ckpt`; the other epochs are deleted.

In [ ]:
DELETE_OTHER_EPOCHS = True

best_models = []
for fold in range(N_FOLDS):
    fold_dir = RUN_DIR / f"fold_{fold}"
    best_path = fold_dir / "best_model.ckpt"
    shutil.move(epoch_checkpoint(fold, BEST_EPOCH), best_path)
    best_models.append(str(best_path))
    if DELETE_OTHER_EPOCHS:
        shutil.rmtree(fold_dir / "checkpoints")

best_folds = cv_metrics[cv_metrics["epoch"] == BEST_EPOCH].set_index("fold")
selection = dict(
    select_metrics=SELECT_METRICS,
    best_epoch=BEST_EPOCH,
    epochs_trained=EPOCHS,
    n_folds=N_FOLDS,
    mean={m: float(best_row[f"{m}_mean"]) for m in metric_names},
    std={m: float(best_row[f"{m}_std"]) for m in metric_names},
    per_fold={m: {int(f): float(v) for f, v in best_folds[m].items()} for m in metric_names},
    models=best_models,
)
(RUN_DIR / "selection.json").write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))

In [ ]:
# Files produced by this run
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(RUN_DIR)):60s} {path.stat().st_size / 1e6:8.2f} MB")